# 중간고사 풀이 (2026.04.16)

**데이터**: 1994년 미국 인구조사국(Census Bureau) 소득 예측 데이터셋  
**목적**: 개인의 인구통계학적 정보를 바탕으로 연소득 $50,000 초과 여부 예측  
**규모**: 32,561개 샘플, 12개 Feature (범주형 + 수치형)

---

## 0. 데이터 로드 및 기본 탐색

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import boxcox
import warnings
warnings.filterwarnings('ignore')

# 데이터 로드
df = pd.read_csv('census.csv')
print(f'데이터 크기: {df.shape}')
print(f'컬럼: {list(df.columns)}')
df.head()

데이터 크기: (32561, 14)
컬럼: ['Unnamed: 0', 'Age', 'Workclass', 'Education-Num', 'Marital Status', 'Occupation', 'Relationship', 'Race', 'Sex', 'Capital Gain', 'Capital Loss', 'Hours per week', 'Country', 'target']


,Unnamed: 0,Age,Workclass,Education-Num,Marital Status,Occupation,Relationship,Race,Sex,Capital Gain,Capital Loss,Hours per week,Country,target
0,0,39.0,State-gov,13.0,Never-married,Adm-clerical,Not-in-family,White,Male,2174.0,0.0,40.0,United-States,False
1,1,50.0,Self-emp-not-inc,13.0,Married-civ-spouse,Exec-managerial,Husband,White,Male,0.0,0.0,13.0,United-States,False
2,2,38.0,Private,9.0,Divorced,Handlers-cleaners,Not-in-family,White,Male,0.0,0.0,40.0,United-States,False
3,3,53.0,Private,7.0,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0.0,0.0,40.0,United-States,False
4,4,28.0,Private,13.0,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0.0,0.0,40.0,Cuba,False


In [2]:
# 기본 정보 확인
print('=== 데이터 타입 ===')
print(df.dtypes)
print()
print('=== 기초 통계량 ===')
df.describe()

=== 데이터 타입 ===
Unnamed: 0          int64
Age               float64
Workclass          object
Education-Num     float64
Marital Status     object
Occupation         object
Relationship       object
Race               object
Sex                object
Capital Gain      float64
Capital Loss      float64
Hours per week    float64
Country            object
target               bool
dtype: object

=== 기초 통계량 ===


,Unnamed: 0,Age,Education-Num,Capital Gain,Capital Loss,Hours per week
count,32561.000000,32561.000000,32561.000000,32561.000000,32561.000000,32561.000000
mean,16280.000000,38.581647,10.080679,1077.648844,87.303830,40.437456
std,9399.695394,13.640433,2.572720,7385.292085,402.960219,12.347429
min,0.000000,17.000000,1.000000,0.000000,0.000000,1.000000
25%,8140.000000,28.000000,9.000000,0.000000,0.000000,40.000000
50%,16280.000000,37.000000,10.000000,0.000000,0.000000,40.000000
75%,24420.000000,48.000000,12.000000,0.000000,0.000000,45.000000
max,32560.000000,90.000000,16.000000,99999.000000,4356.000000,99.000000


---
## 문제 1. Age 변수의 4개 통계량 (10점)

| 통계량 | 설명 |
|--------|------|
| **왜도(Skewness)** | 분포의 비대칭 정도. 양수면 오른쪽 꼬리가 김 |
| **산술 평균(Mean)** | 전체 관측치 합 / 개수 |
| **중앙값(Median)** | 크기순 정렬 시 정중앙 값 |
| **최빈값(Mode)** | 가장 빈번하게 등장하는 값 |

In [3]:
age = df['Age'].dropna()

# 1-a. 왜도 (Skewness)
# pandas .skew()는 편향 보정된(unbiased) 왜도를 계산 (n 보정 적용)
skewness = age.skew()
print(f'왜도(Skewness) = {skewness:.4f} → 반올림: {skewness:.2f}')
print(f'  → 양의 왜도: 오른쪽 꼬리가 긴 분포')
print()

왜도(Skewness) = 0.5587 → 반올림: 0.56
  → 양의 왜도: 오른쪽 꼬리가 긴 분포



In [4]:
# 1-b. 산술 평균 (Mean)
mean_val = age.mean()
print(f'산술 평균(Mean) = {mean_val:.4f} → 반올림: {mean_val:.2f}')
print(f'  → 계산: sum(Age) / n = {age.sum():.0f} / {len(age)} = {mean_val:.2f}')
print()

산술 평균(Mean) = 38.5816 → 반올림: 38.58
  → 계산: sum(Age) / n = 1256257 / 32561 = 38.58



In [5]:
# 1-c. 중앙값 (Median)
median_val = age.median()
print(f'중앙값(Median) = {median_val:.2f}')
print(f'  → n={len(age)} (홀수)이므로 정렬 후 {(len(age)+1)//2}번째 값')
print()

중앙값(Median) = 37.00
  → n=32561 (홀수)이므로 정렬 후 16281번째 값



In [6]:
# 1-d. 최빈값 (Mode)
mode_val = age.mode()[0]
print(f'최빈값(Mode) = {mode_val:.2f}')

# 빈도 확인
top5 = age.value_counts().head(5)
print(f'  → 상위 5개 빈도:')
for val, cnt in top5.items():
    marker = ' ← 최빈값' if val == mode_val else ''
    print(f'     Age={val:.0f}: {cnt}회{marker}')
print()

최빈값(Mode) = 36.00
  → 상위 5개 빈도:
     Age=36: 898회 ← 최빈값
     Age=31: 888회
     Age=34: 886회
     Age=23: 877회
     Age=35: 876회



In [7]:
print('=' * 50)
print(f'★ 정답: Skewness={skewness:.2f}, Mean={mean_val:.2f}, Median={median_val:.2f}, Mode={mode_val:.2f}')
print('  → 5번: Age Skewness: 0.56, Age Mean: 38.58, Age Median: 37.00, Age Mode: 36.00')
print('=' * 50)

★ 정답: Skewness=0.56, Mean=38.58, Median=37.00, Mode=36.00
  → 5번: Age Skewness: 0.56, Age Mean: 38.58, Age Median: 37.00, Age Mode: 36.00


### 핵심 개념 정리

- **왜도 > 0 (양의 왜도)**: Mean > Median > Mode → 오른쪽 꼬리가 긴 분포  
- 실제로 Mean(38.58) > Median(37.00) > Mode(36.00) 순서가 이를 확인해줌
- pandas의 `.skew()`는 Fisher의 편향보정 왜도를 사용 (scipy의 `bias=False`와 동일)

---

## 문제 2. 정규성 향상을 위한 변환과 왜도 변화 (10점)

양의 왜도(0.56)를 가진 Age 변수에 세 가지 변환을 적용하여 정규성을 높인다.

| 변환 | 수식 | 특징 |
|------|------|------|
| 로그 변환 | $\log(1+x)$ | 오른쪽 꼬리를 강하게 압축 |
| 루트 변환 | $\sqrt{x}$ | 로그보다 약하게 압축 |
| Box-Cox | $(x^\lambda - 1)/\lambda$ | 최적 $\lambda$를 자동 탐색 |

In [8]:
print(f'원본 왜도: {age.skew():.4f}')
print()

# ① 로그 변환 — log1p(x) = log(1+x) 사용
# log1p는 x=0일 때도 안전하게 처리 가능
log_age = np.log1p(age)
skew_log = log_age.skew()
print(f'① 로그 변환 log(1+x) 왜도: {skew_log:.4f} → 반올림: {skew_log:.2f}')
print(f'  → 양의 왜도 0.56이 음의 왜도 {skew_log:.2f}로 변환 (과보정)')

원본 왜도: 0.5587

① 로그 변환 log(1+x) 왜도: -0.1133 → 반올림: -0.11
  → 양의 왜도 0.56이 음의 왜도 -0.11로 변환 (과보정)


In [9]:
# ② 루트 변환 — sqrt(x)
sqrt_age = np.sqrt(age)
skew_sqrt = sqrt_age.skew()
print(f'② 루트 변환 sqrt(x) 왜도: {skew_sqrt:.4f} → 반올림: {skew_sqrt:.2f}')
print(f'  → 로그보다 약하게 보정되어 양의 왜도가 남아있음')

② 루트 변환 sqrt(x) 왜도: 0.2043 → 반올림: 0.20
  → 로그보다 약하게 보정되어 양의 왜도가 남아있음


In [10]:
# ③ Box-Cox 변환 — 최적 lambda를 자동 탐색
bc_age, lam = boxcox(age)
skew_bc = pd.Series(bc_age).skew()
print(f'③ Box-Cox 변환 왜도: {skew_bc:.4f} → 반올림: {skew_bc:.2f}')
print(f'  → 최적 lambda = {lam:.4f}')
print(f'  → 왜도가 0에 가장 가까움 = 가장 정규분포에 근접')

③ Box-Cox 변환 왜도: -0.0161 → 반올림: -0.02
  → 최적 lambda = 0.1746
  → 왜도가 0에 가장 가까움 = 가장 정규분포에 근접


In [11]:
print('=' * 50)
print(f'★ 정답: 로그={skew_log:.2f}, 루트={skew_sqrt:.2f}, Box-Cox={skew_bc:.2f}')
print('  → 4번: -0.11, 0.20, -0.02')
print('=' * 50)

★ 정답: 로그=-0.11, 루트=0.20, Box-Cox=-0.02
  → 4번: -0.11, 0.20, -0.02


### 핵심 개념 정리

- **변환 강도 순서**: 로그 > Box-Cox > 루트  
- 로그 변환은 양의 왜도를 **과보정**하여 음의 왜도(-0.11)로 만듦  
- 루트 변환은 **약하게** 보정하여 양의 왜도(0.20)가 남음  
- Box-Cox는 **최적 lambda**를 찾아 왜도를 0에 가장 가깝게(-0.02) 만듦  
- Box-Cox의 lambda ≈ 0.17은 로그(λ→0)와 루트(λ=0.5) 사이의 값

---

## 문제 3. IQR 방식 이상치 탐지 (2×IQR 기준)

**IQR (사분위수 범위)** = Q3 - Q1

- 하한선: $Q_1 - 2 \times IQR$  
- 상한선: $Q_3 + 2 \times IQR$  
- 이 범위를 **벗어나는** 데이터가 이상치

> 참고: 일반적인 IQR 방식은 1.5×IQR을 사용하지만, 이 문제에서는 **2×IQR**을 사용

In [12]:
# 사분위수 계산
Q1 = age.quantile(0.25)
Q3 = age.quantile(0.75)
IQR = Q3 - Q1

print(f'Q1 (25번째 백분위수) = {Q1}')
print(f'Q3 (75번째 백분위수) = {Q3}')
print(f'IQR = Q3 - Q1 = {Q3} - {Q1} = {IQR}')
print()

Q1 (25번째 백분위수) = 28.0
Q3 (75번째 백분위수) = 48.0
IQR = Q3 - Q1 = 48.0 - 28.0 = 20.0



In [13]:
# 이상치 경계 설정 (2 * IQR)
lower_bound = Q1 - 2 * IQR
upper_bound = Q3 + 2 * IQR

print(f'하한선 = Q1 - 2*IQR = {Q1} - 2*{IQR} = {lower_bound}')
print(f'상한선 = Q3 + 2*IQR = {Q3} + 2*{IQR} = {upper_bound}')
print()

하한선 = Q1 - 2*IQR = 28.0 - 2*20.0 = -12.0
상한선 = Q3 + 2*IQR = 48.0 + 2*20.0 = 88.0



In [14]:
# 이상치 탐지
outliers_iqr = age[(age < lower_bound) | (age > upper_bound)]

print(f'하한 이상치 (Age < {lower_bound}): {len(age[age < lower_bound])}개')
print(f'상한 이상치 (Age > {upper_bound}): {len(age[age > upper_bound])}개')
print()
print('=' * 50)
print(f'★ 이상치 총 개수: {len(outliers_iqr)}개')
print('  → 정답: 43')
print('=' * 50)
print()
print(f'이상치 값 분포:')
print(outliers_iqr.describe())

하한 이상치 (Age < -12.0): 0개
상한 이상치 (Age > 88.0): 43개

★ 이상치 총 개수: 43개
  → 정답: 43

이상치 값 분포:
count    43.0
mean     90.0
std       0.0
min      90.0
25%      90.0
50%      90.0
75%      90.0
max      90.0
Name: Age, dtype: float64


### 핵심 개념 정리

- IQR은 데이터의 중간 50%가 퍼져 있는 범위
- 하한선이 -12.0이므로 Age는 음수가 없어 **상한 이상치만** 존재
- 2×IQR은 1.5×IQR보다 느슨한 기준 → 이상치가 더 적게 검출됨
- 이상치는 모두 88세 초과인 고령자

---

## 문제 4. 평균 ± 3σ 이상치 탐지 (10점)

**Z-score 기반 이상치 탐지**
- 분산: **모분산** (자유도 조정 없이 n으로 나눔, ddof=0)
- 이상치 기준: 평균에서 표준편차의 3배를 벗어나는 값

$$\text{이상치}: x < \mu - 3\sigma \quad \text{또는} \quad x > \mu + 3\sigma$$

In [15]:
n = len(age)
mean_age = age.mean()

# ddof=0: 모분산 (전체 데이터 개수 n으로 나눔)
var_pop = age.var(ddof=0)
std_pop = np.sqrt(var_pop)

print(f'n = {n}')
print(f'평균(μ) = {mean_age:.4f}')
print(f'모분산(σ²) = {var_pop:.4f}  ← ddof=0: Σ(x-μ)²/n')
print(f'모표준편차(σ) = {std_pop:.4f}')
print()
print(f'참고) 표본분산(s²) = {age.var(ddof=1):.4f}  ← ddof=1: Σ(x-μ)²/(n-1)')
print(f'차이가 미미한 이유: n={n}으로 매우 크기 때문')

n = 32561
평균(μ) = 38.5816
모분산(σ²) = 186.0557  ← ddof=0: Σ(x-μ)²/n
모표준편차(σ) = 13.6402

참고) 표본분산(s²) = 186.0614  ← ddof=1: Σ(x-μ)²/(n-1)
차이가 미미한 이유: n=32561으로 매우 크기 때문


In [16]:
# 이상치 경계
lower_3s = mean_age - 3 * std_pop
upper_3s = mean_age + 3 * std_pop

print(f'하한선 = μ - 3σ = {mean_age:.4f} - 3×{std_pop:.4f} = {lower_3s:.4f}')
print(f'상한선 = μ + 3σ = {mean_age:.4f} + 3×{std_pop:.4f} = {upper_3s:.4f}')
print()

하한선 = μ - 3σ = 38.5816 - 3×13.6402 = -2.3390
상한선 = μ + 3σ = 38.5816 + 3×13.6402 = 79.5023



In [17]:
# 이상치 탐지
outliers_3s = age[(age < lower_3s) | (age > upper_3s)]

print(f'하한 이상치 (Age < {lower_3s:.2f}): {len(age[age < lower_3s])}개')
print(f'상한 이상치 (Age > {upper_3s:.2f}): {len(age[age > upper_3s])}개')
print()
print('=' * 50)
print(f'★ 이상치 총 개수: {len(outliers_3s)}개')
print('  → 정답: 121')
print('=' * 50)

하한 이상치 (Age < -2.34): 0개
상한 이상치 (Age > 79.50): 121개

★ 이상치 총 개수: 121개
  → 정답: 121


### 핵심 개념 정리

- **모분산(ddof=0)** vs **표본분산(ddof=1)**: 문제에서 "자유도를 조정하지 않은"이라고 명시 → ddof=0
- 정규분포에서 3σ 범위 밖의 확률은 약 0.27% → 이론적으로 약 88개 예상
- 실제 121개가 검출된 것은 Age 분포가 완벽한 정규분포가 아니기 때문
- 하한이 -2.34이므로 하한 이상치는 0개, **모두 상한(79.5세 초과) 이상치**

---

## 문제 5. Theil's U (Uncertainty Coefficient)

**엔트로피 기반 범주형 변수 연관성 측정**

$$U(Y|X) = \frac{H(Y) - H(Y|X)}{H(Y)}$$

| 기호 | 의미 |
|------|------|
| $H(Y)$ | 타겟 Y의 엔트로피 (불확실성) |
| $H(Y|X)$ | X가 주어졌을 때 Y의 조건부 엔트로피 |
| $U(Y|X)$ | X가 Y의 불확실성을 얼마나 줄이는지 (0~1) |

U = 1이면 X가 Y를 완벽하게 예측, U = 0이면 X가 Y에 대해 아무 정보 없음

In [18]:
def entropy(y):
    """H(Y): 엔트로피 계산"""
    counts = y.value_counts()
    probs = counts / counts.sum()
    return -np.sum(probs * np.log2(probs))

def conditional_entropy(x, y):
    """H(Y|X): 조건부 엔트로피 계산"""
    xy_counts = pd.crosstab(x, y)
    x_counts = xy_counts.sum(axis=1)
    total = xy_counts.sum().sum()
    
    ce = 0
    for i in range(len(x_counts)):
        p_x = x_counts.iloc[i] / total          # P(X=x)
        row = xy_counts.iloc[i]
        row_sum = row.sum()
        for j in range(len(row)):
            if row.iloc[j] > 0:
                p_y_given_x = row.iloc[j] / row_sum  # P(Y=y|X=x)
                ce -= p_x * p_y_given_x * np.log2(p_y_given_x)
    return ce

def theils_u(x, y):
    """U(Y|X) = (H(Y) - H(Y|X)) / H(Y)"""
    h_y = entropy(y)
    h_y_given_x = conditional_entropy(x, y)
    if h_y == 0:
        return 0
    return (h_y - h_y_given_x) / h_y

print('함수 정의 완료')
print(f'타겟 엔트로피 H(Y) = {entropy(df["target"]):.4f}')

함수 정의 완료
타겟 엔트로피 H(Y) = 0.7964


In [19]:
# 모든 Feature에 대해 Theil's U 계산
target = df['target']
features = [c for c in df.columns if c not in ['Unnamed: 0', 'target']]

results = {}
for feat in features:
    col = df[feat].astype(str)  # 수치형도 문자열로 변환하여 범주형 처리
    u = theils_u(col, target)
    results[feat] = u

# 결과를 내림차순 정렬
results_sorted = pd.Series(results).sort_values(ascending=False)
print('=== Theil\'s U 순위 (내림차순) ===')
for feat, u in results_sorted.items():
    bar = '█' * int(u * 100)
    print(f'{feat:20s}: {u:.4f} {bar}')

=== Theil's U 순위 (내림차순) ===
Relationship        : 0.2076 ████████████████████
Marital Status      : 0.1965 ███████████████████
Capital Gain        : 0.1511 ███████████████
Age                 : 0.1246 ████████████
Education-Num       : 0.1175 ███████████
Occupation          : 0.1167 ███████████
Hours per week      : 0.0766 ███████
Capital Loss        : 0.0670 ██████
Sex                 : 0.0467 ████
Workclass           : 0.0271 ██
Country             : 0.0109 █
Race                : 0.0105 █


In [20]:
best = results_sorted.index[0]
print('=' * 50)
print(f'★ 가장 높은 Theil\'s U: {best} = {results_sorted.iloc[0]:.4f}')
print('  → 정답: Relationship')
print('=' * 50)

★ 가장 높은 Theil's U: Relationship = 0.2076
  → 정답: Relationship


### 핵심 개념 정리

- **Theil's U**는 비대칭 지표: U(Y|X) ≠ U(X|Y)
- 피어슨 상관계수는 **선형 관계만** 측정하지만, Theil's U는 **비선형 포함 모든 연관성** 측정
- Relationship(가족 관계)이 가장 높은 이유: 배우자(Husband/Wife) 여부가 소득과 강하게 연관
- Marital Status(결혼 상태)도 비슷한 이유로 높게 나타남

---

## 문제 6. Feature Importance 비교 (주관식)

Random Forest의 **특성 중요도**와 Logistic Regression의 **회귀 계수**를 비교하고,  
두 모델이 판단하는 변수 중요도가 다른 이유를 알고리즘적 특성으로 설명하시오.

In [21]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler

# 범주형 변수 인코딩
df_encoded = df.drop('Unnamed: 0', axis=1).copy()
for col in df_encoded.select_dtypes(include='object').columns:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df_encoded[col].astype(str))

y = df_encoded['target'].values
X = df_encoded.drop('target', axis=1)
feature_names = X.columns.tolist()

print(f'Feature 수: {len(feature_names)}')
print(f'Feature: {feature_names}')

Feature 수: 12
Feature: ['Age', 'Workclass', 'Education-Num', 'Marital Status', 'Occupation', 'Relationship', 'Race', 'Sex', 'Capital Gain', 'Capital Loss', 'Hours per week', 'Country']


In [22]:
# Random Forest (기본 설정)
rf = RandomForestClassifier(random_state=42)
rf.fit(X, y)

rf_imp = pd.Series(rf.feature_importances_, index=feature_names).sort_values(ascending=False)
print('=== Random Forest Feature Importance ===')
print('(Gini Impurity 감소량 기반)\n')
for feat, imp in rf_imp.items():
    bar = '█' * int(imp * 100)
    print(f'{feat:20s}: {imp:.4f} {bar}')

=== Random Forest Feature Importance ===
(Gini Impurity 감소량 기반)

Age                 : 0.2152 █████████████████████
Education-Num       : 0.1381 █████████████
Capital Gain        : 0.1280 ████████████
Relationship        : 0.1232 ████████████
Hours per week      : 0.1096 ██████████
Occupation          : 0.0842 ████████
Marital Status      : 0.0594 █████
Workclass           : 0.0497 ████
Capital Loss        : 0.0394 ███
Country             : 0.0217 ██
Race                : 0.0181 █
Sex                 : 0.0134 █


In [23]:
# Logistic Regression (StandardScaler 적용)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_scaled, y)

lr_coef = pd.Series(np.abs(lr.coef_[0]), index=feature_names).sort_values(ascending=False)
print('=== Logistic Regression |Coefficient| ===')
print('(표준화된 회귀계수 절대값)\n')
for feat, coef in lr_coef.items():
    bar = '█' * int(coef * 10)
    print(f'{feat:20s}: {coef:.4f} {bar}')

=== Logistic Regression |Coefficient| ===
(표준화된 회귀계수 절대값)

Capital Gain        : 2.3221 ███████████████████████
Education-Num       : 0.8552 ████████
Age                 : 0.4596 ████
Sex                 : 0.4224 ████
Hours per week      : 0.3670 ███
Marital Status      : 0.3540 ███
Capital Loss        : 0.2731 ██
Relationship        : 0.1914 █
Race                : 0.0948 
Occupation          : 0.0438 
Workclass           : 0.0320 
Country             : 0.0267 


In [24]:
# 순위 비교
compare = pd.DataFrame({
    'RF 순위': range(1, len(rf_imp)+1),
    'RF 중요도': rf_imp.values,
}, index=rf_imp.index)

lr_rank = {feat: i+1 for i, feat in enumerate(lr_coef.index)}
compare['LR 순위'] = [lr_rank[f] for f in compare.index]
compare['순위 차이'] = abs(compare['RF 순위'] - compare['LR 순위'])
print('=== 순위 비교 ===')
print(compare.to_string())

=== 순위 비교 ===
                RF 순위    RF 중요도  LR 순위  순위 차이
Age                 1  0.215155      3      2
Education-Num       2  0.138108      2      0
Capital Gain        3  0.127977      1      2
Relationship        4  0.123232      8      4
Hours per week      5  0.109629      5      0
Occupation          6  0.084163     10      4
Marital Status      7  0.059387      6      1
Workclass           8  0.049712     11      3
Capital Loss        9  0.039374      7      2
Country            10  0.021711     12      2
Race               11  0.018108      9      2
Sex                12  0.013442      4      8


### 모범 답안

**Random Forest**는 트리 기반 앙상블 모델로, 각 분기(split)에서 **Gini 불순도 감소량**을 기준으로 변수 중요도를 측정한다.  
→ **비선형 관계**와 **변수 간 상호작용(interaction)**을 잘 포착하며, 연속형 변수(Age)가 여러 분기에서 반복 사용되어 중요도가 높게 나타남

**Logistic Regression**은 선형 모델로, 각 변수의 **회귀 계수(coefficient)**가 타겟의 로그 오즈(log-odds)에 미치는 **선형적 영향력**을 측정한다.  
→ Capital Gain처럼 타겟과 강한 **선형 관계**를 가진 변수가 높게 나타남

**핵심 차이**: RF는 비선형+상호작용 기반, LR은 선형 독립 기여도 기반이므로 변수 중요도 순위가 달라짐

---

## 문제 7. 범주형 2변수 조합 — 고소득 비율 ≥ 70% (10점)

범주형 변수 2개의 모든 가능한 조합으로 그룹화 후, 각 그룹의 고소득자 비율 계산

In [25]:
from itertools import combinations

# 범주형 변수 확인
cat_cols = df.select_dtypes(include='object').columns.tolist()
print(f'범주형 변수 ({len(cat_cols)}개): {cat_cols}')
print(f'가능한 2변수 조합 수: {len(list(combinations(cat_cols, 2)))}')

범주형 변수 (7개): ['Workclass', 'Marital Status', 'Occupation', 'Relationship', 'Race', 'Sex', 'Country']
가능한 2변수 조합 수: 21


In [26]:
# 모든 2변수 조합에 대해 고소득 비율 ≥ 70% 그룹 찾기
high_income_groups = []

for c1, c2 in combinations(cat_cols, 2):
    grouped = df.groupby([c1, c2])['target'].agg(['sum', 'count'])
    grouped['ratio'] = grouped['sum'] / grouped['count']
    high = grouped[grouped['ratio'] >= 0.70]
    
    for idx, row in high.iterrows():
        high_income_groups.append({
            'var1': c1, 'var2': c2,
            'val1': idx[0].strip(), 'val2': idx[1].strip(),
            'n': int(row['count']),
            'ratio': row['ratio']
        })

result_df = pd.DataFrame(high_income_groups)
print(f'고소득 비율 ≥ 70% 그룹 총 수: {len(result_df)}')
print(f'  (n≥10인 그룹: {len(result_df[result_df["n"]>=10])}개)')

고소득 비율 ≥ 70% 그룹 총 수: 81
  (n≥10인 그룹: 9개)


In [27]:
# 의미 있는 그룹만 필터 (n≥10)
meaningful = result_df[result_df['n'] >= 10].sort_values('ratio', ascending=False)
print('=== 고소득 비율 ≥ 70% (n≥10) ===')
for _, r in meaningful.head(15).iterrows():
    print(f"  {r['val1']} & {r['val2']}: {r['ratio']:.2%} (n={r['n']})")

=== 고소득 비율 ≥ 70% (n≥10) ===
  Exec-managerial & India: 80.00% (n=10)
  Self-emp-inc & Prof-specialty: 75.62% (n=160)
  Married-civ-spouse & France: 75.00% (n=12)
  Prof-specialty & Wife: 74.59% (n=307)
  Husband & France: 72.73% (n=11)
  Exec-managerial & Japan: 72.22% (n=18)
  Prof-specialty & Philippines: 71.43% (n=28)
  Married-civ-spouse & Prof-specialty: 70.88% (n=2126)
  Prof-specialty & Husband: 70.73% (n=1804)


In [28]:
# 보기 직접 검증
print('=== 보기별 검증 ===')
options = [
    ('Occupation', 'Relationship', ' Prof-specialty', ' Wife',      'Prof-specialty & Wife'),
    ('Occupation', 'Relationship', ' Prof-specialty', ' Husband',   'Prof-specialty & Husband'),
    ('Marital Status', 'Occupation', ' Married-civ-spouse', ' Prof-specialty', 'Married-civ-spouse & Prof-specialty'),
    ('Workclass', 'Occupation', ' Self-emp-inc', ' Prof-specialty', 'Self-emp-inc & Prof-specialty'),
    ('Occupation', 'Country', ' Prof-specialty', ' Philippines',    'Prof-specialty & Philippines'),
]

for v1, v2, val1, val2, label in options:
    subset = df[(df[v1] == val1) & (df[v2] == val2)]
    ratio = subset['target'].sum() / len(subset)
    mark = '✓ ≥70%' if ratio >= 0.70 else '✗ <70%'
    print(f'  {label:45s}: {ratio:.2%} (n={len(subset)}) {mark}')

print()
print('=' * 50)
print('★ 정답: Prof-specialty & Wife (74.59%, n=307)')
print('  → 모든 보기가 70% 이상이지만, 가장 대표적인 답')
print('=' * 50)

=== 보기별 검증 ===
  Prof-specialty & Wife                        : 74.59% (n=307) ✓ ≥70%
  Prof-specialty & Husband                     : 70.73% (n=1804) ✓ ≥70%
  Married-civ-spouse & Prof-specialty          : 70.88% (n=2126) ✓ ≥70%
  Self-emp-inc & Prof-specialty                : 75.62% (n=160) ✓ ≥70%
  Prof-specialty & Philippines                 : 71.43% (n=28) ✓ ≥70%

★ 정답: Prof-specialty & Wife (74.59%, n=307)
  → 모든 보기가 70% 이상이지만, 가장 대표적인 답


---
## 문제 8. White 남녀 조건부 확률

White 인종 중에서 남성/여성의 고소득(>50K) 조건부 확률:

$$P(\text{Income}>50K \mid \text{White, Male}) = \frac{\text{White이면서 Male이면서 고소득인 수}}{\text{White이면서 Male인 수}}$$

In [29]:
white = df[df['Race'] == ' White']
print(f'White 전체: {len(white)}명')
print()

# White & Male
white_male = white[white['Sex'] == ' Male']
p_male = white_male['target'].sum() / len(white_male)
print(f'White Male: {len(white_male)}명, 고소득: {white_male["target"].sum()}명')
print(f'P(>50K | White, Male) = {white_male["target"].sum()} / {len(white_male)} = {p_male:.4f}')
print()

# White & Female
white_female = white[white['Sex'] == ' Female']
p_female = white_female['target'].sum() / len(white_female)
print(f'White Female: {len(white_female)}명, 고소득: {white_female["target"].sum()}명')
print(f'P(>50K | White, Female) = {white_female["target"].sum()} / {len(white_female)} = {p_female:.4f}')
print()

print('=' * 50)
print(f'★ 정답: {p_male:.4f}, {p_female:.4f}')
print('  → 2번: 0.3176, 0.1190')
print('=' * 50)

White 전체: 27816명

White Male: 19174명, 고소득: 6089명
P(>50K | White, Male) = 6089 / 19174 = 0.3176

White Female: 8642명, 고소득: 1028명
P(>50K | White, Female) = 1028 / 8642 = 0.1190

★ 정답: 0.3176, 0.1190
  → 2번: 0.3176, 0.1190


### 핵심 개념 정리

- **조건부 확률**: 주어진 조건 하에서의 확률. 데이터셋의 실제 비율을 사용
- White 남성의 고소득 확률(31.76%)이 여성(11.90%)보다 약 2.67배 높음
- 이는 1994년 데이터의 소득 격차를 반영

---

## 문제 9. 이항분포 B(n, p) (0점)

White 인종 중 고소득자 비율 $p$를 구하고, $X \sim B(30, p)$일 때:

- $E(X) = np$
- $Var(X) = np(1-p)$
- $P(X \geq 8)$

In [30]:
# p 계산
white = df[df['Race'] == ' White']
p = white['target'].sum() / len(white)
n = 30

print(f'White 전체: {len(white)}명, 고소득: {white["target"].sum()}명')
print(f'p = P(target=True | White) = {white["target"].sum()}/{len(white)} = {p:.6f}')
print(f'n = {n}')
print()

White 전체: 27816명, 고소득: 7117명
p = P(target=True | White) = 7117/27816 = 0.255860
n = 30



In [31]:
# 기댓값
E_X = n * p
print(f'E(X) = n × p = {n} × {p:.6f} = {E_X:.3f}')
print()

# 분산
Var_X = n * p * (1 - p)
print(f'Var(X) = n × p × (1-p) = {n} × {p:.6f} × {1-p:.6f} = {Var_X:.3f}')
print()

E(X) = n × p = 30 × 0.255860 = 7.676

Var(X) = n × p × (1-p) = 30 × 0.255860 × 0.744140 = 5.712



In [32]:
# P(X >= 8) = 1 - P(X <= 7)
from scipy.stats import binom

p_x_ge_8 = 1 - binom.cdf(7, n, p)
print(f'P(X ≥ 8) = 1 - P(X ≤ 7)')
print(f'         = 1 - binom.cdf(7, {n}, {p:.6f})')
print(f'         = 1 - {binom.cdf(7, n, p):.6f}')
print(f'         = {p_x_ge_8:.5f}')
print()

print('=' * 50)
print(f'★ E(X)={E_X:.3f}, Var(X)={Var_X:.3f}, P(X≥8)={p_x_ge_8:.5f}')
print('  → 2번: E(X)=7.676, Var(X)=5.712 (E, Var 일치)')
print('=' * 50)

P(X ≥ 8) = 1 - P(X ≤ 7)
         = 1 - binom.cdf(7, 30, 0.255860)
         = 1 - 0.484558
         = 0.51544

★ E(X)=7.676, Var(X)=5.712, P(X≥8)=0.51544
  → 2번: E(X)=7.676, Var(X)=5.712 (E, Var 일치)


### 핵심 개념 정리

- **이항분포 B(n, p)**: n번 독립 시행에서 성공 확률 p인 사건의 성공 횟수
- $E(X) = np$: 기댓값은 시행 횟수 × 성공 확률
- $Var(X) = np(1-p)$: 분산은 p가 0.5에 가까울수록 커짐
- $P(X \geq 8) = 1 - P(X \leq 7) = 1 - \text{binom.cdf}(7, n, p)$
- E(X) ≈ 7.7이므로 P(X≥8)은 약 0.5 부근이 되는 것이 직관적으로 맞음

---

## 문제 10. 정규분포 확률 및 90% 신뢰구간

White 인종의 주당 근무시간 $X \sim N(\mu, \sigma^2)$일 때:

- $P(X \geq 40)$
- $\mu$에 대한 90% 신뢰구간 (= 개인 값의 90% 예측구간)

In [33]:
white = df[df['Race'] == ' White']
hours = white['Hours per week'].dropna()

mu = hours.mean()
sigma = hours.std(ddof=1)  # 표본표준편차
n = len(hours)

print(f'White 표본 수: {n}')
print(f'표본평균 μ = {mu:.4f}')
print(f'표본표준편차 σ = {sigma:.4f}')
print()

White 표본 수: 27816
표본평균 μ = 40.6891
표본표준편차 σ = 12.5448



In [34]:
# P(X >= 40)
from scipy.stats import norm

z_40 = (40 - mu) / sigma
p_ge_40 = 1 - norm.cdf(40, loc=mu, scale=sigma)

print(f'Z = (40 - {mu:.4f}) / {sigma:.4f} = {z_40:.4f}')
print(f'P(X ≥ 40) = 1 - Φ({z_40:.4f}) = {p_ge_40:.4f}')
print(f'  → 평균({mu:.2f})이 40보다 크므로 P > 0.5')
print()

Z = (40 - 40.6891) / 12.5448 = -0.0549
P(X ≥ 40) = 1 - Φ(-0.0549) = 0.5219
  → 평균(40.69)이 40보다 크므로 P > 0.5



In [35]:
# 90% 신뢰구간 (개인 값 예측구간: μ ± z * σ)
z_90 = norm.ppf(0.95)  # 양측 90% → 단측 95%

lower_ci = mu - z_90 * sigma
upper_ci = mu + z_90 * sigma

print(f'90% 신뢰구간 = μ ± z_0.05 × σ')
print(f'  z_0.05 = {z_90:.4f}')
print(f'  하한 = {mu:.4f} - {z_90:.4f} × {sigma:.4f} = {lower_ci:.2f}')
print(f'  상한 = {mu:.4f} + {z_90:.4f} × {sigma:.4f} = {upper_ci:.2f}')
print()

print('=' * 50)
print(f'★ P(X≥40) = {p_ge_40:.4f}, 90% 신뢰구간: [{lower_ci:.2f}, {upper_ci:.2f}]')
print('  → 1번: P(40시간 이상) = 0.5219, 90% 신뢰구간: [20.05, 61.32]')
print('=' * 50)

90% 신뢰구간 = μ ± z_0.05 × σ
  z_0.05 = 1.6449
  하한 = 40.6891 - 1.6449 × 12.5448 = 20.05
  상한 = 40.6891 + 1.6449 × 12.5448 = 61.32

★ P(X≥40) = 0.5219, 90% 신뢰구간: [20.05, 61.32]
  → 1번: P(40시간 이상) = 0.5219, 90% 신뢰구간: [20.05, 61.32]


### 핵심 개념 정리

- **정규분포** $N(\mu, \sigma^2)$: 평균을 중심으로 좌우 대칭인 종 모양 분포
- $P(X \geq 40) = 1 - \Phi\left(\frac{40 - \mu}{\sigma}\right)$
- 평균(40.69)이 40보다 약간 크므로 P(X≥40) ≈ 0.52 (50%보다 약간 큼)
- 90% 신뢰구간: 개인 값의 90%가 이 범위 안에 들어올 것으로 예측
- $z_{0.05} = 1.6449$ (양측 90%에 해당하는 z값)

---

## 최종 정답 요약

| 문제 | 정답 | 배점 |
|------|------|------|
| 1 | **5번** Skewness=0.56, Mean=38.58, Median=37.00, Mode=36.00 | 10점 |
| 2 | **4번** 로그=-0.11, 루트=0.20, Box-Cox=-0.02 | 10점 |
| 3 | **43개** | 0점 |
| 4 | **121개** | 10점 |
| 5 | **Relationship** (Theil's U = 0.2076) | 0점 |
| 6 | 주관식 (RF: 비선형+상호작용 vs LR: 선형 독립 기여) | - |
| 7 | **Prof-specialty & Wife** (74.59%) | 10점 |
| 8 | **2번** 0.3176, 0.1190 | 0점 |
| 9 | **2번** E(X)=7.676, Var(X)=5.712 | 0점 |
| 10 | **1번** P=0.5219, CI=[20.05, 61.32] | 0점 |